# M1 — Validación de Detección de Placas (YOLOv9-t ONNX) + OCR Híbrido
Reutiliza el modelo de placas de proyectos anteriores y compara **fast-plate-ocr**, **EasyOCR** y el pipeline **Híbrido (con validación de formato boliviano)** para reportar métricas reales en el informe técnico.

In [ ]:
!nvidia-smi -L || echo "Ejecutando en CPU (ONNXRuntime funciona en CPU y GPU)"
!pip install -q onnxruntime fast-plate-ocr[onnx] easyocr opencv-python-headless pandas matplotlib

In [ ]:
import os, re, time, json, glob
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import onnxruntime as ort
from fast_plate_ocr import LicensePlateRecognizer
import easyocr

# Si subes los pesos ONNX desde /inferencia/models a /content/models o clonas el repo:
MODEL_DIR = Path('/content/models')
if not MODEL_DIR.exists():
    MODEL_DIR = Path('models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('Carpeta de modelos:', MODEL_DIR.resolve())

## 1. Subir modelos ONNX de M1 y dataset de prueba (imágenes + ground_truth.csv)
El CSV debe tener columnas: `filename,placa_real` (ej. `auto_01.jpg,2481KLM`).

In [ ]:
from google.colab import files
print('Sube yolo-v9-t-384-license-plates-end2end.onnx, cct_xs_v2_global.onnx, cct_xs_v2_global_plate_config.yaml (si no estan ya en /content/models):')
uploaded = files.upload()
for name in uploaded:
    dst = MODEL_DIR / name
    os.replace(name, dst)
    print('Guardado:', dst)

In [ ]:
DET_PATH = MODEL_DIR / 'yolo-v9-t-384-license-plates-end2end.onnx'
OCR_PATH = MODEL_DIR / 'cct_xs_v2_global.onnx'
OCR_CFG = MODEL_DIR / 'cct_xs_v2_global_plate_config.yaml'

assert DET_PATH.exists(), f'Falta {DET_PATH}'
assert OCR_PATH.exists() and OCR_CFG.exists(), 'Faltan pesos de fast-plate-ocr'

LADO = 384
FORMATO_BOLIVIA = re.compile(r'^\d{3,4}[A-Z]{3}$')
PERMITIDOS = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'

providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if 'CUDAExecutionProvider' in ort.get_available_providers() else ['CPUExecutionProvider']
sesion_det = ort.InferenceSession(str(DET_PATH), providers=providers)
lector_fast = LicensePlateRecognizer(onnx_model_path=str(OCR_PATH), plate_config_path=str(OCR_CFG))
lector_easy = easyocr.Reader(['en'], gpu=True if 'CUDAExecutionProvider' in providers else False, verbose=False)
print('M1 inicializado con providers:', providers)

In [ ]:
def limpiar_placa(texto: str) -> str:
    return re.sub(r'[^A-Z0-9]', '', (texto or '').upper())

def detectar_placas(imagen_bgr, umbral=0.35):
    alto, ancho = imagen_bgr.shape[:2]
    escala = min(LADO / alto, LADO / ancho)
    nw, nh = round(ancho * escala), round(alto * escala)
    mx, my = (LADO - nw) / 2, (LADO - nh) / 2
    lienzo = cv2.resize(imagen_bgr, (nw, nh), interpolation=cv2.INTER_LINEAR)
    lienzo = cv2.copyMakeBorder(lienzo, round(my - 0.1), round(my + 0.1), round(mx - 0.1), round(mx + 0.1), cv2.BORDER_CONSTANT, value=(114, 114, 114))
    entrada = (lienzo[:, :, ::-1].transpose(2, 0, 1) / 255.0).astype(np.float32)[None]
    salida = sesion_det.run(None, {sesion_det.get_inputs()[0].name: entrada})[0]
    cajas = []
    for _, x1, y1, x2, y2, _, conf in salida:
        if conf < umbral:
            continue
        x1, x2 = [(v - mx) / escala for v in (x1, x2)]
        y1, y2 = [(v - my) / escala for v in (y1, y2)]
        cajas.append((int(max(0, x1)), int(max(0, y1)), int(min(ancho, x2)), int(min(alto, y2)), float(conf)))
    cajas.sort(key=lambda c: -c[4])
    return cajas

def leer_ocr_fast(recorte_bgr):
    rgb = cv2.cvtColor(recorte_bgr, cv2.COLOR_BGR2RGB)
    pred = lector_fast.run_one(rgb, return_confidence=True)
    txt = limpiar_placa(pred.plate)
    conf = float(pred.char_probs.mean()) if pred.char_probs is not None else 0.0
    return txt, conf

def leer_ocr_easy(recorte_bgr):
    lecturas = lector_easy.readtext(recorte_bgr, allowlist=PERMITIDOS)
    if lecturas:
        altura = lambda l: max(p[1] for p in l[0]) - min(p[1] for p in l[0])
        hmax = max(altura(l) for l in lecturas)
        lecturas = [l for l in lecturas if altura(l) >= hmax * 0.6]
    lecturas.sort(key=lambda l: l[0][0][0])
    txt = limpiar_placa(''.join(t for _, t, _ in lecturas))
    conf = float(sum(c for _, _, c in lecturas) / len(lecturas)) if lecturas else 0.0
    return txt, conf

def leer_ocr_hibrido(recorte_bgr):
    t_fast, c_fast = leer_ocr_fast(recorte_bgr)
    t_easy, c_easy = leer_ocr_easy(recorte_bgr)
    bol_fast = bool(FORMATO_BOLIVIA.match(t_fast))
    bol_easy = bool(FORMATO_BOLIVIA.match(t_easy))
    if bol_easy and not bol_fast:
        return t_easy, c_easy, 'easyocr'
    if bol_fast and not bol_easy:
        return t_fast, c_fast, 'fast-plate-ocr'
    if c_easy > c_fast and len(t_easy) >= 5:
        return t_easy, c_easy, 'easyocr'
    return (t_fast, c_fast, 'fast-plate-ocr') if len(t_fast) >= len(t_easy) else (t_easy, c_easy, 'easyocr')

## 2. Evaluación sobre imágenes de prueba
Sube un `.zip` con fotos de vehículos/placas y opcionalmente un `ground_truth.csv` para calcular exactitud OCR y CER.

In [ ]:
DIR_TEST = Path('/content/test_placas')
DIR_TEST.mkdir(parents=True, exist_ok=True)
print('Sube un .zip con imagenes de prueba y/o ground_truth.csv:')
for fname in files.upload():
    if fname.lower().endswith('.zip'):
        !unzip -qo "$fname" -d /content/test_placas
    else:
        os.replace(fname, DIR_TEST / fname)

def levenshtein(a, b):
    dp = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        ndp = [i]
        for j, cb in enumerate(b, 1):
            ndp.append(min(ndp[-1] + 1, dp[j] + 1, dp[j - 1] + (0 if ca == cb else 1)))
        dp = ndp
    return dp[-1]

gt_path = DIR_TEST / 'ground_truth.csv'
gt_map = {}
if gt_path.exists():
    df_gt = pd.read_csv(gt_path)
    gt_map = {str(r['filename']).strip(): limpiar_placa(str(r['placa_real'])) for _, r in df_gt.iterrows()}

EXT = ('.jpg', '.jpeg', '.png', '.webp')
imgs = sorted(p for p in DIR_TEST.rglob('*') if p.suffix.lower() in EXT)
filas = []
for p in imgs:
    img = cv2.imread(str(p))
    if img is None:
        continue
    t0 = time.perf_counter()
    cajas = detectar_placas(img)
    dt_det = (time.perf_counter() - t0) * 1000
    real = gt_map.get(p.name, '')
    if not cajas:
        filas.append({'archivo': p.name, 'detectada': False, 'real': real, 'fast': '', 'easy': '', 'hibrido': '', 'exacto': False, 'cer': 1.0 if real else None, 'ms': round(dt_det, 1)})
        continue
    x1, y1, x2, y2, cdet = cajas[0]
    crop = img[y1:y2, x1:x2]
    t1 = time.perf_counter()
    tf, cf = leer_ocr_fast(crop)
    te, ce = leer_ocr_easy(crop)
    th, ch, ganador = leer_ocr_hibrido(crop)
    dt_tot = dt_det + (time.perf_counter() - t1) * 1000
    cer = levenshtein(th, real) / max(1, len(real)) if real else None
    filas.append({
        'archivo': p.name, 'detectada': True, 'conf_det': round(cdet, 3),
        'real': real, 'fast': tf, 'easy': te, 'hibrido': th, 'ganador': ganador,
        'conf_ocr': round(ch, 3), 'exacto': (th == real) if real else None,
        'cer': round(cer, 3) if cer is not None else None, 'ms': round(dt_tot, 1)
    })

df_res = pd.DataFrame(filas)
display(df_res)
resumen = {
    'total_imagenes': int(len(df_res)),
    'tasa_deteccion_placa': round(float(df_res['detectada'].mean()), 4) if len(df_res) else 0,
    'latencia_promedio_ms': round(float(df_res['ms'].mean()), 2) if len(df_res) else 0,
}
if gt_map and len(df_res):
    eval_df = df_res[df_res['real'] != '']
    resumen['exactitud_ocr_exact_match'] = round(float(eval_df['exacto'].mean()), 4)
    resumen['cer_promedio'] = round(float(eval_df['cer'].mean()), 4)
print(json.dumps(resumen, indent=2, ensure_ascii=False))
with open('/content/resumen_metricas_m1.json', 'w') as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)

## 3. Diagnóstico de entorno (Evidencia para bonificación conda/mamba/venv)

In [ ]:
import sys, platform, torch, ultralytics
diag = {
    'servicio': 'colab_m1_validacion',
    'python': sys.version.split()[0],
    'os': platform.platform(),
    'torch': torch.__version__,
    'cuda_disponible': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'onnxruntime_providers': ort.get_available_providers(),
}
print(json.dumps(diag, indent=2, ensure_ascii=False))